# Arbol B
El código desarrollado construye la estructura y el comportamiento de un Árbol B estándar, permitiendo el almacenamiento ordenado de datos, su búsqueda y la preservación del balance estructural ante inserciones y eliminaciones.

### 1. Estructura Base
Para construir el árbol, se definió primero la unidad fundamental a través de la clase **`NodoB`**. Cada nodo cuenta con:
*   **`claves`**: Un arreglo dinámico que almacena los valores ordenados.
*   **`hijos`**: Un arreglo de referencias (punteros) que conectan con los nodos del nivel inferior.
*   **`es_hoja`**: Una variable booleana que identifica si el nodo se encuentra en el último nivel del árbol, lo cual es crucial para detener la recursividad.

### 2. Controlador Central y Reglas (`ArbolB`)
La clase **`ArbolB`** administra la jerarquía. Al instanciarse, recibe el `orden` del árbol (cantidad máxima de hijos por nodo). A partir de este parámetro, se construyeron las funciones `min_claves()` y `max_claves()` que calculan dinámicamente los límites permitidos para detonar las operaciones de balanceo.

### 3. Mecanismo de Búsqueda
La función de búsqueda se implementó mediante un recorrido recursivo descendente.
*   Se integró la librería nativa `bisect` para aplicar **búsqueda binaria** (específicamente `bisect_left`) dentro del arreglo de `claves` de cada nodo. Esto permite localizar instantáneamente la posición de un valor o deducir la rama por la cual descender.
*   Al encontrar el dato, se construyó un retorno en formato de diccionario que entrega la clave, el nivel de profundidad en el que fue hallada y la clasificación del nodo (raíz, interno u hoja).

### 4. Lógica de Inserción y División
Para la inserción de nuevos datos, el algoritmo se diseñó para descender siempre hasta un nodo hoja.
*   **Inserción:** El dato se posiciona en el lugar correcto manteniendo el orden.
*   **División (`_dividir_nodo`):** Tras insertar, se evalúa si el nodo excedió el límite de claves. De ser así, se emplea la técnica de partición (slicing) de arreglos para dividir las claves y los hijos exactamente a la mitad.
*   **Promoción:** La función de división retorna una tupla que contiene la clave central y el nuevo nodo derecho creado. El sistema captura estos datos en el nivel superior y los inserta en el nodo padre. Si la raíz original se divide, se crea una nueva raíz superior.

### 5. Lógica de Eliminación y Reestructuración
El proceso de eliminación es la parte más detallada del código, diseñada para cubrir múltiples casos y evitar que los nodos queden vacíos:
*   **Borrado Directo y por Sucesor:** Si la clave está en una hoja, se extrae de la lista. Si se encuentra en un nodo interno, se programó una búsqueda del "sucesor" (el valor más pequeño del subárbol derecho) mediante la función `_obtener_minimo`. La clave a borrar se reemplaza por el sucesor, y luego se elimina el sucesor original de la hoja.
*   **Balanceo (`_balancear`):** Tras cualquier borrado, el sistema evalúa si el nodo quedó con menos claves que el mínimo permitido. Para restaurarlo, se implementaron dos estrategias:
    1.  **Rotaciones (`_rotar_izquierda`, `_rotar_derecha`):** Si un nodo hermano adyacente tiene claves suficientes, se realiza un préstamo. Una clave del hermano sube al nodo padre, y la clave separadora del padre baja al nodo deficiente.
    2.  **Fusión (`_fusionar`):** Si ningún hermano adyacente tiene claves para prestar, el sistema agrupa el nodo deficiente, la clave separadora del padre y el nodo hermano en un único bloque continuo, reduciendo la altura del árbol si es necesario.

### Ejemplo Práctico de Funcionamiento (Árbol B de Orden 3)

Para ilustrar la ejecución del algoritmo, se expone el caso de un Árbol B inicializado con un orden de 3. De acuerdo con los parámetros matemáticos, cada nodo almacenará un máximo de 2 claves y un mínimo de 1 clave, con excepción del nodo raíz al inicio del proceso.

#### 1. Inserciones Iniciales
Se procede con la inserción de los valores `10` y `20`. Ambos elementos se ubican secuencialmente en el nodo raíz inicial, el cual opera temporalmente como nodo hoja.
*   **Estado del Árbol:**
    ```text
    [Nivel 0] Claves: [10, 20]
    ```

#### 2. División por saturación de capacidad
Se introduce el valor `30`. El nodo raíz alcanza un total de 3 claves (`[10, 20, 30]`), superando el límite máximo establecido (2).
El método `_dividir_nodo` extrae el valor central (`20`) y lo promueve. Al tratarse de la raíz, se instancia un nuevo nivel superior. Los valores restantes se asignan a los nodos hijos izquierdo y derecho de forma equilibrada.
*   **Estado del Árbol:**
    ```text
    [Nivel 0] Claves: [20]
      [Nivel 1] Claves: [10]
      [Nivel 1] Claves: [30]
    ```

#### 3. Expansión horizontal
Se inserta el valor `40`. El sistema determina mediante búsqueda binaria que `40` es estrictamente mayor que `20`, por lo que desciende por la rama derecha y aloja el valor en la hoja correspondiente.
*   **Estado del Árbol:**
    ```text
    [Nivel 0] Claves: [20]
      [Nivel 1] Claves: [10]
      [Nivel 1] Claves: [30, 40]
    ```

#### 4. Ejecución de búsqueda
Al invocar la función de búsqueda para el valor `40`, el algoritmo compara el objetivo con la raíz (`20`), desciende por el hijo derecho, localiza el dato y emite un reporte técnico con las propiedades del hallazgo:
*   **Retorno de la función:** `{"clave": 40, "nivel": 1, "tipo": "hoja"}`

#### 5. Eliminación y balanceo por rotación
Se emite la instrucción de eliminar el valor `10`.
Dicha clave es removida de la hoja izquierda, dejando al nodo con cero elementos, lo cual infringe la restricción del mínimo requerido (1 clave). En respuesta, el sistema activa el método `_balancear`. Al detectar que el nodo hermano derecho (`[30, 40]`) posee claves suficientes para realizar una cesión, se ejecuta una operación de rotación a través del padre:
1.  La clave separadora ubicada en la raíz (`20`) desciende para abastecer la hoja izquierda deficiente.
2.  La primera clave del nodo hermano derecho (`30`) asciende y se establece como el nuevo separador en la raíz.
*   **Estado Final del Árbol:**
    ```text
    [Nivel 0] Claves: [30]
      [Nivel 1] Claves: [20]
      [Nivel 1] Claves: [40]
    ```

In [13]:
import math
import bisect

class NodoB:
    def __init__(self, es_hoja=True):
        self.claves = []
        self.hijos = []
        self.es_hoja = es_hoja

class ArbolB:
    def __init__(self, orden):
        self.orden = orden
        self.raiz = NodoB(es_hoja=True)

    def min_claves(self):
        return math.ceil(self.orden / 2) - 1

    def max_claves(self):
        return self.orden - 1

    # BÚSQUEDA
    def buscar(self, clave, nodo=None, nivel=0):
        if nodo is None and nivel == 0:
            nodo = self.raiz

        idx = bisect.bisect_left(nodo.claves, clave)

        if idx < len(nodo.claves) and nodo.claves[idx] == clave:
            tipo = "raiz" if nodo == self.raiz else ("hoja" if nodo.es_hoja else "interno")
            return {"clave": clave, "nivel": nivel, "tipo": tipo}

        if nodo.es_hoja:
            return None

        return self.buscar(clave, nodo.hijos[idx], nivel + 1)

    # INSERCIÓN
    def insertar(self, clave):
        if self.buscar(clave) is not None:
            print(f"La clave {clave} ya existe, omitiendo.")
            return

        division = self._insertar_recursivo(self.raiz, clave)
        if division:
            clave_media, nodo_derecho = division
            nueva_raiz = NodoB(es_hoja=False)
            nueva_raiz.claves = [clave_media]
            nueva_raiz.hijos = [self.raiz, nodo_derecho]
            self.raiz = nueva_raiz

        print(f"Clave {clave} insertada.")

    def _insertar_recursivo(self, nodo, clave):
        idx = bisect.bisect_left(nodo.claves, clave)

        if nodo.es_hoja:
            nodo.claves.insert(idx, clave)
            if len(nodo.claves) > self.max_claves():
                return self._dividir_nodo(nodo)
            return None

        division = self._insertar_recursivo(nodo.hijos[idx], clave)
        if division:
            clave_media, nodo_derecho = division
            nodo.claves.insert(idx, clave_media)
            nodo.hijos.insert(idx + 1, nodo_derecho)
            if len(nodo.claves) > self.max_claves():
                return self._dividir_nodo(nodo)
        return None

    def _dividir_nodo(self, nodo):
        medio = len(nodo.claves) // 2
        clave_media = nodo.claves[medio]
        nuevo_nodo = NodoB(es_hoja=nodo.es_hoja)
        nuevo_nodo.claves = nodo.claves[medio + 1:]
        nodo.claves = nodo.claves[:medio]

        if not nodo.es_hoja:
            nuevo_nodo.hijos = nodo.hijos[medio + 1:]
            nodo.hijos = nodo.hijos[:medio + 1]

        return (clave_media, nuevo_nodo)

    # ELIMINACIÓN
    def eliminar(self, clave):
        if not self.buscar(clave):
            print(f"Clave {clave} no encontrada.")
            return

        self._eliminar_recursivo(self.raiz, clave)

        if len(self.raiz.claves) == 0 and not self.raiz.es_hoja:
            self.raiz = self.raiz.hijos[0]
        print(f"Clave {clave} eliminada.")

    def _eliminar_recursivo(self, nodo, clave):
        idx = bisect.bisect_left(nodo.claves, clave)

        if idx < len(nodo.claves) and nodo.claves[idx] == clave:
            if nodo.es_hoja:
                nodo.claves.pop(idx)
            else:
                sucesor = self._obtener_minimo(nodo.hijos[idx + 1])
                nodo.claves[idx] = sucesor
                self._eliminar_recursivo(nodo.hijos[idx + 1], sucesor)
                self._balancear(nodo, idx + 1)
            return True

        if nodo.es_hoja:
            return False

        se_elimino = self._eliminar_recursivo(nodo.hijos[idx], clave)
        if se_elimino:
            self._balancear(nodo, idx)
        return se_elimino

    def _obtener_minimo(self, nodo):
        while not nodo.es_hoja:
            nodo = nodo.hijos[0]
        return nodo.claves[0]

    def _balancear(self, padre, idx_hijo):
        hijo = padre.hijos[idx_hijo]
        if len(hijo.claves) >= self.min_claves():
            return

        if idx_hijo > 0 and len(padre.hijos[idx_hijo - 1].claves) > self.min_claves():
            self._rotar_derecha(padre, idx_hijo)
        elif idx_hijo < len(padre.hijos) - 1 and len(padre.hijos[idx_hijo + 1].claves) > self.min_claves():
            self._rotar_izquierda(padre, idx_hijo)
        else:
            if idx_hijo > 0:
                self._fusionar(padre, idx_hijo - 1)
            else:
                self._fusionar(padre, idx_hijo)

    def _rotar_derecha(self, padre, idx_hijo):
        hijo = padre.hijos[idx_hijo]
        hermano_izq = padre.hijos[idx_hijo - 1]

        hijo.claves.insert(0, padre.claves[idx_hijo - 1])
        padre.claves[idx_hijo - 1] = hermano_izq.claves.pop()

        if not hijo.es_hoja:
            hijo.hijos.insert(0, hermano_izq.hijos.pop())

    def _rotar_izquierda(self, padre, idx_hijo):
        hijo = padre.hijos[idx_hijo]
        hermano_der = padre.hijos[idx_hijo + 1]

        hijo.claves.append(padre.claves[idx_hijo])
        padre.claves[idx_hijo] = hermano_der.claves.pop(0)

        if not hijo.es_hoja:
            hijo.hijos.append(hermano_der.hijos.pop(0))

    def _fusionar(self, padre, idx_izq):
        hijo_izq = padre.hijos[idx_izq]
        hijo_der = padre.hijos[idx_izq + 1]

        hijo_izq.claves.append(padre.claves.pop(idx_izq))
        hijo_izq.claves.extend(hijo_der.claves)

        if not hijo_izq.es_hoja:
            hijo_izq.hijos.extend(hijo_der.hijos)

        padre.hijos.pop(idx_izq + 1)

    # VISUALIZACIÓN
    def imprimir(self, nodo=None, nivel=0):
        if nodo is None and nivel == 0:
            nodo = self.raiz
            print("Estructura del Árbol B:")

        if nodo is None:
            return

        identacion = "  " * nivel
        print(f"{identacion}[Nivel {nivel}] Claves: {nodo.claves}")

        if not nodo.es_hoja:
            for hijo in nodo.hijos:
                self.imprimir(hijo, nivel + 1)


# MENÚ INTERACTIVO
def main():
    while True:
        try:
            orden = int(input("Ingrese el orden del arbol (M >= 3): "))
            if orden >= 3: break
            print("El orden debe ser al menos 3.")
        except ValueError:
            print("Entrada invalida.")

    arbol = ArbolB(orden)

    while True:
        print("\n1. Insertar  2. Eliminar  3. Buscar  4. Imprimir  0. Salir")
        opc = input("Seleccione: ")

        try:
            if opc == '1':
                arbol.insertar(int(input("Clave a insertar: ")))
            elif opc == '2':
                arbol.eliminar(int(input("Clave a eliminar: ")))
            elif opc == '3':
                res = arbol.buscar(int(input("Clave a buscar: ")))
                if res:
                    print(f"Encontrada: {res['clave']} (Nivel {res['nivel']}, {res['tipo']})")
                else:
                    print("Clave no encontrada.")
            elif opc == '4':
                arbol.imprimir()
            elif opc == '0':
                break
            else:
                print("Opción inválida.")
        except ValueError:
            print("Por favor ingrese números válidos.")

if __name__ == "__main__":
    main()

Ingrese el orden del arbol (M >= 3): 3

1. Insertar  2. Eliminar  3. Buscar  4. Imprimir  0. Salir
Seleccione: 1
Clave a insertar: 10
Clave 10 insertada.

1. Insertar  2. Eliminar  3. Buscar  4. Imprimir  0. Salir
Seleccione: 1
Clave a insertar: 20
Clave 20 insertada.

1. Insertar  2. Eliminar  3. Buscar  4. Imprimir  0. Salir
Seleccione: 4
Estructura del Árbol B:
[Nivel 0] Claves: [10, 20]

1. Insertar  2. Eliminar  3. Buscar  4. Imprimir  0. Salir
Seleccione: 1
Clave a insertar: 30
Clave 30 insertada.

1. Insertar  2. Eliminar  3. Buscar  4. Imprimir  0. Salir
Seleccione: 4
Estructura del Árbol B:
[Nivel 0] Claves: [20]
  [Nivel 1] Claves: [10]
  [Nivel 1] Claves: [30]

1. Insertar  2. Eliminar  3. Buscar  4. Imprimir  0. Salir
Seleccione: 1
Clave a insertar: 40
Clave 40 insertada.

1. Insertar  2. Eliminar  3. Buscar  4. Imprimir  0. Salir
Seleccione: 4
Estructura del Árbol B:
[Nivel 0] Claves: [20]
  [Nivel 1] Claves: [10]
  [Nivel 1] Claves: [30, 40]

1. Insertar  2. Eliminar  3. 

KeyboardInterrupt: Interrupted by user